[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/fast_track/05_classes_basics.ipynb)

# 🏎️ Notebook 5 (fast track) — Classes & OOP basics

> 🏎️ **You're on the fast track.** This is a focused version of the canonical [`01_foundations/06_classes_and_oop.ipynb`](../01_foundations/06_classes_and_oop.ipynb). It teaches **normal (hand-written) classes in depth** — attributes, methods, and inheritance — and shows how classes are used in **data science** (a `Dataset` wrapper, a `fit`/`transform` transformer hierarchy, and a `fit`/`predict` model). The special *dunder* methods (`__repr__`, `__eq__`) and `@dataclass` shortcut are left for the canonical version — open it once you want that material.

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a class is and how it differs from a dict, a function, and an instance.
2. **Define** a class with `__init__`, instance **attributes**, and **methods** — including the `self` parameter.
3. **Distinguish** *instance attributes* from *class attributes*, and know when to reach for each.
4. **Write** methods that read, mutate, and call *other* methods on `self`.
5. **Use inheritance** — define a base class and extend it with subclasses that override or add behaviour (`super()`, `is-a`).
6. **Apply classes to data science** — a `Dataset` wrapper, a `BaseTransformer` → scaler hierarchy with `fit`/`transform`, and an estimator with `fit`/`predict`.

**Prerequisites:** NB 1–4. You should be comfortable with functions, default arguments, and dicts.

**Time budget:** ~60 minutes.


## 1. Why classes? — when functions + dicts aren't enough

Imagine you're tracking a customer for a small SaaS business. You start with a dict:


In [1]:
customer = {
    "name": "Acme Inc.",
    "monthly_revenue": 14823.4567,
    "is_active": True,
}
print(customer["name"])


Acme Inc.


That works fine. But the *behaviour* you want to attach (compute annual revenue, deactivate the account, check whether the customer is high-value) ends up scattered across free-standing functions:


In [4]:
def annual_revenue(c):
    return c["monthly_revenue"] * 12

def deactivate(c):
    c["is_active"] = False

def is_high_value(c, threshold=10_000):
    return c["monthly_revenue"] >= threshold

print(annual_revenue(customer))         # 177881.4804
print(is_high_value(customer))          # True


177881.4804
True


Two problems start to bite as the codebase grows:

1. **The data and the behaviour drift apart.** Six months later someone adds a new field `monthly_revenue_eur` and forgets to update `annual_revenue`. Now the function silently returns the wrong number for European customers.
2. **Every helper has to repeat `customer["..."]`.** Typos in those string keys are silent — `customer["montly_revenue"]` raises `KeyError` only at runtime.

A **class** solves both: it gives you one place that says *"a customer has these attributes and supports these behaviours"*. Everything that operates on a customer lives inside it.


## 2. 🧠 Mental model — class as blueprint, instance as concrete thing

The single most important sentence about classes:

> **A class is a blueprint. An instance is a concrete thing built from that blueprint.**

```
  class Customer:           ← THE BLUEPRINT — defined once
      __init__(name, ...)       (the rules for building one)
      annual_revenue()          (a behaviour every customer can do)

          │  build one
          ▼
  ┌─────────────────────┐    ┌─────────────────────┐    ┌─────────────────────┐
  │  Customer instance  │    │  Customer instance  │    │  Customer instance  │
  │  ─────────────────  │    │  ─────────────────  │    │  ─────────────────  │
  │  name = "Acme"      │    │  name = "Beta"      │    │  name = "Cara"      │
  │  rev  = 14823.4567  │    │  rev  = 980.00      │    │  rev  = 50000.00    │
  └─────────────────────┘    └─────────────────────┘    └─────────────────────┘
       acme.annual...              beta.annual...              cara.annual...
```

Three separate **instances**, all built from the same **class**. Each instance has its own data; all of them share the same behaviour. Anything you change about the *blueprint* is automatically reflected in all current and future instances.


## 3. Defining a class — `__init__` and `self`

Here is the same customer record written as a class:


In [5]:
class Customer:
    """A small SaaS customer."""

    def __init__(self, name, monthly_revenue, is_active=True):
        # Constructor — runs when you write `Customer(...)`.
        self.name             = name
        self.monthly_revenue  = monthly_revenue
        self.is_active        = is_active

    def annual_revenue(self):
        return self.monthly_revenue * 12

    def deactivate(self):
        self.is_active = False

    def is_high_value(self, threshold=10_000):
        return self.monthly_revenue >= threshold

acme = Customer("Acme Inc.", 14823.4567)
print(acme.name)                  # Acme Inc.
print(acme.annual_revenue())      # 177881.4804
print(acme.is_high_value())       # True
acme.deactivate()
print(acme.is_active)             # False


Acme Inc.
177881.4804
True
False


Three syntactic things to absorb:

1. **`class ClassName:`** opens the blueprint. By convention class names are `CapitalCase`.
2. **`__init__(self, ...)`** is the **constructor**. Python calls it for you when you write `Customer(...)`. Its job is to set up the instance's initial state by assigning to `self.something`.
3. **`self`** is the *instance the method is being called on*. When you write `acme.annual_revenue()`, Python silently rewrites that to `Customer.annual_revenue(acme)` — `acme` becomes `self` inside the method body.

> 🎯 **Intuition.** `self` looks weird the first ten times. Read it as **"this particular instance"**. `self.name` means "the `name` attribute of *this* customer", as opposed to `name` (a local variable).


### 🔬 What actually happens when you write `Customer(...)`?

`acme = Customer("ACME Corp", 10_000)` looks like one step, but Python does four. Watching them is what makes `self` finally click:

**Step 1 — build an empty object.** Python creates a blank `Customer` with no data yet:

```text
Customer object
---------------
name            = ?
monthly_revenue = ?
```

**Step 2 — call the constructor for you.** Python passes the brand-new object in as the first argument. This:

```python
acme = Customer("ACME Corp", 10_000)
```

is really:

```python
Customer.__init__(<the new object>, "ACME Corp", 10_000)
```

So inside `__init__`, **`self` is that new object.**

**Step 3 — run the assignments.** `self.name = name` and `self.monthly_revenue = monthly_revenue` execute as:

```python
<the new object>.name            = "ACME Corp"
<the new object>.monthly_revenue = 10_000
```

**Step 4 — hand the finished object back.** The result is stored in your variable:

```text
acme
 ├── name            = "ACME Corp"
 └── monthly_revenue = 10_000
```

That's the whole life cycle: *create blank → `__init__` fills it via `self` → you get it back.*


### Two different things called `name` — parameter vs attribute

Look closely at the constructor — the word `name` appears twice, and they are **not the same thing**:

```python
def __init__(self, name):   # ← (1) the PARAMETER
    self.name = name        # ← (2) the ATTRIBUTE   = (1) the parameter
```

| | `name` (parameter) | `self.name` (attribute) |
|---|---|---|
| What it is | A **local variable** of `__init__` | A piece of the **object's own state** |
| Lives where | Only inside the constructor | Inside the object |
| Lifespan | **Disappears** when `__init__` finishes | **Persists** for the object's whole life |
| After creation | `name` is gone | `acme.name` still works |

The line `self.name = name` is the bridge: it **copies the temporary parameter into a permanent attribute on the object.** After the constructor returns, the parameter `name` is gone — but `acme.name` lives on inside `acme`.


### `self` is not a keyword — it's just a parameter name

The biggest surprise for beginners: **`self` is not special syntax.** It's an ordinary parameter, and *you* named it. The only reason it's always called `self` is universal convention — Python itself doesn't care. The cell below proves it by naming the first parameter `banana` instead:


In [ ]:
# ⚠️ Educational ONLY — this works, but NEVER write real code like this.
class CustomerWeird:
    def __init__(banana, name):     # "banana" plays the role of self
        banana.name = name          # store on the object that was passed in

c = CustomerWeird("ACME Corp")
print(c.name)                       # ACME Corp — works fine!

# The lesson: `self` is just the *name* of the first parameter — the object
# Python automatically passes in. By convention we always call it `self`.


### Behind the scenes — calling a method

The same automatic-first-argument rule powers *every* method call, not just `__init__`. When you call a method on an object, Python rewrites it and passes the object in as `self`. Watch `acme.annual_revenue()` expand all the way to a number:

```text
acme.annual_revenue()
        │   Python rewrites the call, passing acme in as self
        ▼
Customer.annual_revenue(acme)
        │   inside the method, self = acme
        ▼
return self.monthly_revenue * 12
        │   substitute self → acme
        ▼
return acme.monthly_revenue * 12
        │   acme.monthly_revenue is 10_000
        ▼
return 10_000 * 12
        ▼
120_000
```

So `self` inside `annual_revenue` is simply **whichever object you called the method on**. Call it on `acme`, `self` is `acme`; call it on `globex`, `self` is `globex`. Same method, different object — different answer. The cell below proves both the rewrite *and* the identity:


In [ ]:
acme   = Customer("ACME Corp",  10_000)
globex = Customer("Globex Inc",  5_000)

# 1️⃣ The two call forms are EQUIVALENT — Python rewrites the first into the second:
print(acme.annual_revenue())            # 120000
print(Customer.annual_revenue(acme))    # 120000  ← self=acme made explicit, same result

# 2️⃣ Same method, different object, different data:
print(globex.annual_revenue())          # 60000

# 3️⃣ `self` really IS the object you called the method on — prove it with `is`:
class Probe:
    def who_am_i(self):
        return self                     # hand back whatever self is

p = Probe()
print(p.who_am_i() is p)                # True — `self` inside the method is literally `p`


### Visualising memory — each object carries its own state

After creating two customers, memory holds **two independent objects**. The method is shared (it lives on the class, once); the *data* is separate (it lives on each instance):

```text
   acme                              globex
    │                                  │
    ▼                                  ▼
 +------------------------+      +------------------------+
 | Customer object        |      | Customer object        |
 |------------------------|      |------------------------|
 | name            = ACME |      | name            = Globex|
 | monthly_revenue = 10000|      | monthly_revenue = 5000 |
 +------------------------+      +------------------------+
            \                         /
             \                       /
              ▼                     ▼
        +-------------------------------+
        | Customer class (the blueprint)|
        |  annual_revenue(self) — ONE   |
        |  copy, shared by both objects |
        +-------------------------------+
```

This is the payoff of `self`: when you call `globex.annual_revenue()`, Python passes `globex` in as `self`, so the *one* shared method reaches into the *right* object's data.

> 🎯 **The mental model to keep.** Every time you read `self.something`, translate it in your head to **"the `something` stored inside *this particular* object"**. `self.name` → "this object's name"; `self.monthly_revenue` → "this object's revenue". That one habit makes every class you'll ever read instantly legible.


## 4. Attributes in depth — instance vs class attributes

An **attribute** is just a variable that lives on an object. You read it with `obj.attribute` and set it with `obj.attribute = value`. There are two kinds, and the difference matters:

- **Instance attributes** — assigned with `self.x = ...` inside `__init__` (or any method). **Each instance gets its own copy.** `acme.name` and `beta.name` are independent.
- **Class attributes** — assigned *directly in the class body*, outside any method. **There is one copy, shared by every instance** (and reachable on the class itself). Use them for constants and defaults that are the same for all instances.


In [6]:
class Customer:
    # ── class attributes: one shared copy for ALL customers ──
    company   = "Acme SaaS Inc."        # a shared constant
    count     = 0                       # a shared counter

    def __init__(self, name, monthly_revenue):
        # ── instance attributes: a fresh copy PER customer ──
        self.name            = name
        self.monthly_revenue = monthly_revenue
        Customer.count += 1             # bump the shared counter

a = Customer("North Ltd.", 1200)
b = Customer("South Ltd.", 3400)

print(a.name, "|", b.name)        # North Ltd. | South Ltd.  (independent)
print(a.company, "|", b.company)  # Acme SaaS Inc. | Acme SaaS Inc.  (shared)
print(Customer.count)             # 2 — one shared counter, bumped twice


North Ltd. | South Ltd.
Acme SaaS Inc. | Acme SaaS Inc.
2


## 5. Methods in depth — behaviour that uses `self`

A **method** is a function defined inside a class. The one mechanical difference from a free function is that it automatically receives the instance as its first argument, `self`. Through `self` a method can:

- **read** the instance's attributes (`self.monthly_revenue`),
- **mutate** them (`self.is_active = False`),
- and **call other methods** on the same instance (`self.annual_revenue()`).

That last point is the powerful one: methods compose. Build small methods, then let bigger methods call them — if you change one, every caller stays correct.


In [ ]:
class Invoice:
    TAX_RATE = 0.20                       # class attribute: shared VAT rate

    def __init__(self, net_amount):
        self.net_amount = net_amount

    def tax(self):                        # small method — one job
        return self.net_amount * self.TAX_RATE

    def gross(self):                      # bigger method — REUSES tax()
        return self.net_amount + self.tax()

    def describe(self):                   # bigger still — reuses gross()
        return f"Invoice: net {self.net_amount:.2f}, gross {self.gross():.2f}"

inv = Invoice(100.0)
print(inv.tax())        # 20.0
print(inv.gross())      # 120.0
print(inv.describe())   # Invoice: net 100.00, gross 120.00


Notice `gross()` calls `self.tax()` rather than re-deriving the tax. If the VAT rate or the tax formula ever changes, you edit **one** method and `gross()` + `describe()` follow automatically. This *"call your own methods"* habit is the single biggest reason classes stay maintainable as they grow.


### ⚠️ Common beginner pitfalls

Three mistakes that bite everyone once:

1. **Forgetting `self`** in a method signature — Python will complain that the method needs an argument it didn't get.
2. **Using `=` to set an attribute without going through `self`** — `name = "Acme"` inside a method creates a local variable that disappears when the method exits. The instance is unchanged; you meant `self.name = "Acme"`.
3. **Mutable default arguments** — never write `def __init__(self, tags=[]):`. The list is created *once* and shared across all instances (the same trap as a mutable class attribute). Use `tags=None` and `self.tags = tags or []` inside the body instead.


## 6. Methods vs free functions — which to use?

Use a **method** when the function:

- Operates on the instance's state (`self.something`).
- Belongs naturally to the concept (an *Order* `total()`s itself; an *Order* doesn't need an external `compute_order_total(order)`).

Use a **free function** when it:

- Operates on multiple unrelated objects with equal weight (a `merge(a, b)` doesn't belong inside `a`).
- Is genuinely standalone (parsing, formatting, mathematical utilities).

> 💡 **A useful heuristic.** If you find yourself writing `do_X(thing)` and `do_Y(thing)` and `do_Z(thing)`, those probably want to become `thing.x()`, `thing.y()`, `thing.z()` methods.


## 7. Inheritance — extending an existing class

Sometimes you want a class that does *what an existing class does, plus a bit extra* (or *differently*). That's **inheritance**: a **subclass** (child) inherits all the attributes and methods of a **base class** (parent), and may add new ones or **override** existing ones.

The relationship to look for is **"is-a"**: a `Dog` *is an* `Animal`, a `StandardScaler` *is a* `Transformer`. If you can't say "X is a kind of Y", inheritance is the wrong tool.


In [ ]:
class Animal:
    """Base class — the shape every animal shares."""

    def __init__(self, name):
        self.name = name

    def speak(self):
        # Base says: "every subclass must provide its own voice."
        raise NotImplementedError("subclasses must implement speak()")

    def describe(self):
        # Inherited by all subclasses; it calls speak(), which each overrides.
        return f"{self.name} says {self.speak()}"


class Dog(Animal):                 # Dog IS-A Animal
    def speak(self):
        return "Woof"


class Cat(Animal):                 # Cat IS-A Animal
    def speak(self):
        return "Meow"

for animal in [Dog("Rex"), Cat("Mittens")]:
    print(animal.describe())       # uses INHERITED describe() + OVERRIDDEN speak()


Three observations worth absorbing:

1. **`class Dog(Animal):`** — the parent class goes in parentheses. `Dog` now has everything `Animal` has (the `__init__`, the `describe`) **plus** its own `speak`.
2. **`raise NotImplementedError`** in the base class is how you say *"every subclass must provide its own"*. If you forget to write `speak` in a subclass, calling it raises a clear error instead of doing the wrong thing silently.
3. **`describe()` is written once** in the base class and *reused* by every subclass — yet it calls `self.speak()`, so each subclass plugs in its own behaviour. Write-once, vary-the-details: that's the payoff of inheritance.


### Extending — not just replacing — with `super()`

Often a subclass wants to *add to* the parent's behaviour, not throw it away. `super()` lets a subclass call the parent's version of a method — most commonly the parent's `__init__`, so the subclass can set up the shared state and then add its own:


In [ ]:
class Account:
    def __init__(self, owner):
        self.owner   = owner
        self.balance = 0.0

    def summary(self):
        return f"{self.owner}: {self.balance:.2f}"


class SavingsAccount(Account):                 # SavingsAccount IS-A Account
    def __init__(self, owner, rate):
        super().__init__(owner)                # run Account.__init__ first...
        self.rate = rate                       # ...then add subclass-specific state

    def add_interest(self):                    # brand-new behaviour
        self.balance += self.balance * self.rate

    def summary(self):                         # OVERRIDE, but reuse the parent's
        return super().summary() + f" (rate {self.rate:.0%})"

s = SavingsAccount("Ada", 0.05)
s.balance = 1000
s.add_interest()
print(s.summary())     # Ada: 1050.00 (rate 5%)


`super().__init__(owner)` runs the parent constructor so you don't repeat `self.owner = owner`; `super().summary()` reuses the parent's string and appends to it. **Reuse, don't copy.**

> ⚠️ **Don't over-use inheritance.** A common beginner mistake is to subclass everything. Use inheritance only when the relationship is genuinely *X is a kind of Y*. For "X *uses* a Y" or "X *has* a Y", prefer **composition** — store the helper as an attribute (`self.helper = helper`) — which is more flexible and easier to test.


## 8. Classes in data science — the patterns you'll actually use

Here's why this notebook matters for the rest of the course: **the data science libraries you'll use are built out of classes**, and they follow two patterns over and over. Once you can read these, pandas, scikit-learn, and PyTorch stop looking like magic.

### Pattern 1 — a `Dataset` wrapper (data + behaviour in one place)

Instead of passing a bare list of rows around and writing helper functions for it, wrap it in a class. The data and the operations on it live together — exactly the lesson from Section 1, now applied to a feature matrix.


In [ ]:
class Dataset:
    """A tiny wrapper around feature rows + a target column."""

    def __init__(self, X, y, feature_names):
        self.X = X                       # list of rows, each a list of floats
        self.y = y                       # list of targets
        self.feature_names = feature_names

    def n_samples(self):
        return len(self.X)

    def n_features(self):
        return len(self.feature_names)

    def feature_mean(self, name):
        j = self.feature_names.index(name)         # column index
        col = [row[j] for row in self.X]
        return sum(col) / len(col)

    def summary(self):
        means = {n: round(self.feature_mean(n), 2) for n in self.feature_names}
        return f"Dataset({self.n_samples()} samples, {self.n_features()} features, means={means})"


data = Dataset(
    X=[[25, 50_000], [32, 64_000], [47, 120_000], [51, 98_000]],
    y=[0, 0, 1, 1],
    feature_names=["age", "income"],
)
print(data.summary())                 # one object that knows its own shape & stats
print("mean income:", data.feature_mean("income"))


`data.summary()` calls `data.n_samples()`, `data.n_features()`, and `data.feature_mean()` — methods composing methods, just like `Invoice` in Section 5. This is conceptually what a pandas `DataFrame` is: a class wrapping your data, exposing `.shape`, `.mean()`, `.describe()` as methods.


### Pattern 2 — a transformer hierarchy with `fit` / `transform` (inheritance!)

scikit-learn's preprocessing objects (`StandardScaler`, `MinMaxScaler`, …) all share the **same interface**: `.fit(X)` learns parameters *from the data* and stores them on `self`; `.transform(X)` applies them. That shared shape is a perfect job for a **base class**, with each specific scaler as a **subclass**.


In [ ]:
import numpy as np

class BaseTransformer:
    """Defines the fit / transform / fit_transform contract once, for all scalers."""

    def fit(self, X):
        raise NotImplementedError("subclasses learn their parameters here")

    def transform(self, X):
        raise NotImplementedError("subclasses apply their parameters here")

    def fit_transform(self, X):
        # Written ONCE in the base — every subclass inherits it for free.
        self.fit(X)
        return self.transform(X)


class StandardScaler(BaseTransformer):
    """(x - mean) / std, per column.  IS-A BaseTransformer."""

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)        # learned params get a trailing _
        self.std_  = X.std(axis=0)
        return self                        # return self so you can chain .fit(X).transform(X)

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.std_


class MinMaxScaler(BaseTransformer):
    """(x - min) / (max - min), per column.  Also IS-A BaseTransformer."""

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.min_) / (self.max_ - self.min_)


X = [[25, 50_000], [32, 64_000], [47, 120_000], [51, 98_000]]

print("StandardScaler:\n", StandardScaler().fit_transform(X).round(2))
print("\nMinMaxScaler:\n",  MinMaxScaler().fit_transform(X).round(2))


Look at what inheritance bought us: **`fit_transform` is written exactly once** in `BaseTransformer`, yet both scalers get it. Each subclass only supplies the two methods that genuinely differ (`fit`, `transform`). Add a third scaler tomorrow and you write two methods, not five. This is — almost line for line — how scikit-learn's real transformers are structured (they inherit `fit_transform` from a shared `TransformerMixin`).


### Pattern 3 — an estimator with `fit` / `predict`

Models follow the sibling convention: `.fit(X, y)` learns from labelled data; `.predict(X)` produces predictions. Here's the simplest possible model — a baseline that always predicts the mean of the training targets — written in the exact shape a real sklearn estimator uses.


In [ ]:
class MeanRegressor:
    """A baseline model: predict the average training target for everything."""

    def fit(self, X, y):
        self.prediction_ = float(np.mean(y))   # the one thing it "learns"
        return self

    def predict(self, X):
        # one prediction per input row
        return np.full(len(X), self.prediction_)


X_train = [[1], [2], [3], [4]]
y_train = [10, 20, 30, 40]

model = MeanRegressor().fit(X_train, y_train)
print("learned mean:", model.prediction_)        # 25.0
print("predictions :", model.predict([[99], [100]]))   # [25. 25.]


Every sklearn model — `LinearRegression`, `RandomForestClassifier`, `KMeans` — is a class with this same `fit` / `predict` shape. Because they all share the interface, you can swap one for another without changing the surrounding code:

```python
model = StandardScaler().fit_transform(X)   # preprocess
clf   = SomeClassifier().fit(X, y)          # train  — same .fit() everywhere
preds = clf.predict(X_new)                  # use    — same .predict() everywhere
```

> 🧠 **Mental model.** "Estimator with `fit`/`predict`" and "transformer with `fit`/`transform`" are *conventions*, not language features — but because the whole ecosystem agrees on them, classes that follow the convention drop straight into pipelines, grid searches, and cross-validation. Writing your own helper as a class with the same method names is how you make it *feel native* to the data-science stack.


## 9. 🧠 Mini-recap

- A **class** is a blueprint; an **instance** is a concrete thing built from the blueprint.
- `__init__(self, ...)` is the constructor. `self` is the current instance.
- **Instance attributes** (`self.x = …`) are per-instance; **class attributes** (in the class body) are shared by all instances — never make a *mutable* one by accident.
- **Methods** take `self`, can read/mutate attributes, and should call *other* methods on `self` to stay maintainable.
- **Inheritance** lets a subclass reuse a base class and override or extend it (`super()`); use it only for genuine *is-a* relationships.
- In data science, classes show up as **data wrappers** (`Dataset`, `DataFrame`) and as **`fit`/`transform`** and **`fit`/`predict`** objects — the building blocks of scikit-learn.


## 🧪 Practice exercises

Try each exercise yourself first. Solutions are right below — but you learn while struggling, not while reading the answer.

### Exercise 1 — ⭐ A `Person` class

Define a class `Person` with attributes `name` and `age`, and a method `greet()` that returns the string `"Hi, I'm <name> and I am <age> years old."`.

Create a `Person("Ada", 36)` and print `p.greet()`.

In [ ]:
# Your code here  👇
class Person:
    ...

# p = Person("Ada", 36)
# print(p.greet())


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age  = age

    def greet(self):
        return f"Hi, I'm {self.name} and I am {self.age} years old."

p = Person("Ada", 36)
print(p.greet())
```

**Reasoning.** Three things to internalise. (1) `__init__` is the constructor — it assigns the arguments to `self.name` and `self.age`, which become *instance attributes*. (2) `greet(self)` takes `self` as its first parameter; Python supplies it automatically when you write `p.greet()`. (3) The f-string inside the method body uses `self.name`, not `name` — `name` alone would be an undefined local variable.
</details>

### Exercise 2 — ⭐⭐ A `BankAccount` with deposit and withdraw

Define a `BankAccount` class with attribute `balance` (default `0.0`) and two methods:

- `deposit(amount)` adds `amount` to the balance.
- `withdraw(amount)` subtracts `amount` from the balance **only if** the resulting balance would be ≥ 0;   otherwise it should print `"Insufficient funds"` and leave the balance unchanged.

Demonstrate both methods on a fresh account.

In [ ]:
# Your code here  👇
class BankAccount:
    ...

# a = BankAccount()
# a.deposit(100); print(a.balance)
# a.withdraw(30); print(a.balance)
# a.withdraw(999); print(a.balance)


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class BankAccount:
    def __init__(self, balance=0.0):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if self.balance - amount < 0:
            print("Insufficient funds")
            return
        self.balance -= amount

a = BankAccount()
a.deposit(100); print(a.balance)        # 100.0
a.withdraw(30); print(a.balance)        # 70.0
a.withdraw(999); print(a.balance)       # 'Insufficient funds' then 70.0
```

**Reasoning.** This exercise builds two habits. (1) **Default values in `__init__`** let callers leave out arguments — `BankAccount()` and `BankAccount(50.0)` both work. (2) **Guard at the top, mutate at the bottom.** The `if self.balance - amount < 0: return` shape is more readable than a nested `if/else`.
</details>

### Exercise 3 — ⭐⭐ Class attribute vs instance attribute

Define a `Circle` class where:

- `pi` is a **class attribute** equal to `3.14159` (shared by every circle).
- `radius` is an **instance attribute** set in `__init__`.
- `area()` returns `pi * radius ** 2` using `self.pi` and `self.radius`.

Create two circles with different radii and print each area. Then print `Circle.pi` to confirm it lives on the class.

In [ ]:
# Your code here  👇
class Circle:
    ...

# c1 = Circle(1); c2 = Circle(2)
# print(c1.area(), c2.area())
# print(Circle.pi)


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Circle:
    pi = 3.14159                 # class attribute — one shared copy

    def __init__(self, radius):
        self.radius = radius     # instance attribute — per circle

    def area(self):
        return self.pi * self.radius ** 2

c1 = Circle(1); c2 = Circle(2)
print(c1.area(), c2.area())      # 3.14159 12.56636
print(Circle.pi)                 # 3.14159 — reachable on the class itself
```

**Reasoning.** `pi` is the same for every circle, so it belongs on the **class**, not duplicated into every instance. `radius` differs per circle, so it's an **instance** attribute. Inside `area()`, `self.pi` works because attribute lookup checks the instance first, then falls back to the class — so a constant defined once on the class is visible through every instance.
</details>

### Exercise 4 — ⭐⭐ Inheritance — `Shape`, `Rectangle`, `Square`

Build a small hierarchy:

- `Shape` is a base class with a method `area()` that does `raise NotImplementedError`.
- `Rectangle(Shape)` takes `width` and `height` in `__init__` and overrides `area()`.
- `Square(Rectangle)` takes a single `side` and reuses `Rectangle`'s `area()` by calling `super().__init__(side, side)`.

Confirm `Square(4).area()` returns `16`.

In [ ]:
# Your code here  👇
class Shape:
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Shape:
    def area(self):
        raise NotImplementedError("subclasses must implement area()")

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width  = width
        self.height = height
    def area(self):
        return self.width * self.height

class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)   # a square IS-A rectangle with equal sides

print(Rectangle(3, 4).area())   # 12
print(Square(4).area())         # 16 — inherits Rectangle.area() unchanged
```

**Reasoning.** `Square` writes **no** `area()` of its own — it inherits `Rectangle`'s. All it does is constrain construction (`side, side`) via `super().__init__`. This is the cleanest kind of inheritance: the subclass reuses the parent's behaviour wholesale and only specialises construction. Note the *is-a* chain reads correctly: a square is a rectangle is a shape.
</details>

## 🧠 Stretch exercises

Two deeper exercises that tie classes to data science. Try them yourself before opening the solution.

### Stretch exercise A — ⭐⭐⭐ An `Order` with line items

Build an `Order` class that supports:

- An empty order at construction (`self.items = []` — *not* a mutable default argument).
- `add_item(name, price, qty)` — appends a line item (store each as a dict).
- `total()` — returns the sum of `price * qty` across all items.
- `describe()` — returns a string like `"Order: 2 items, total $49.47"` that **reuses** `total()`.

In [ ]:
# Your code here  👇
class Order:
    ...

# o = Order()
# o.add_item("widget", 9.99, 3)
# o.add_item("gadget", 19.50, 1)
# print(o.describe())     # Order: 2 items, total $49.47


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
class Order:
    def __init__(self):
        self.items = []   # one fresh list per instance — NOT a mutable default arg

    def add_item(self, name, price, qty):
        self.items.append({"name": name, "price": price, "qty": qty})

    def total(self):
        return sum(item["price"] * item["qty"] for item in self.items)

    def describe(self):
        return f"Order: {len(self.items)} items, total ${self.total():.2f}"

o = Order()
o.add_item("widget", 9.99, 3)
o.add_item("gadget", 19.50, 1)
print(o.describe())     # Order: 2 items, total $49.47
```

**Reasoning.** Two important habits. (1) **`self.items = []` inside `__init__`** — *never* `def __init__(self, items=[])`. The latter shares one list across every instance, the same trap as a mutable class attribute (Section 4). (2) **Reuse your own methods** — `describe()` calls `self.total()` instead of recomputing the sum, so the formatting stays consistent if `total()` ever changes.
</details>

### Stretch exercise B — ⭐⭐⭐ A `Standardizer` transformer (data science)

Re-implement a one-column standardizer in the scikit-learn `fit`/`transform` style, inheriting the `fit_transform` convenience method from a base class:

- `BaseTransformer` has `fit`, `transform` (both `raise NotImplementedError`) and a concrete `fit_transform(X)` that calls `self.fit(X)` then `self.transform(X)`.
- `Standardizer(BaseTransformer)` overrides `fit(X)` to store `self.mean_` and `self.std_` (use the `statistics` module or plain Python), and `transform(X)` to return `[(x - mean) / std for x in X]`.

Confirm that `Standardizer().fit_transform([10, 20, 30])` is centred on 0.

In [ ]:
# Your code here  👇
class BaseTransformer:
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import statistics

class BaseTransformer:
    def fit(self, X):
        raise NotImplementedError
    def transform(self, X):
        raise NotImplementedError
    def fit_transform(self, X):          # written once — inherited by every subclass
        self.fit(X)
        return self.transform(X)

class Standardizer(BaseTransformer):
    def fit(self, X):
        self.mean_ = statistics.mean(X)          # learned params: trailing _
        self.std_  = statistics.pstdev(X)
        return self
    def transform(self, X):
        return [(x - self.mean_) / self.std_ for x in X]

print(Standardizer().fit_transform([10, 20, 30]))   # [-1.2247..., 0.0, 1.2247...]
```

**Reasoning.** This is the real scikit-learn shape in miniature. (1) **`fit` learns and stores** parameters on `self` with a trailing-underscore name (`mean_`, `std_`) — the library-wide convention for "computed during fit". (2) **`transform` only uses** those stored params, so you can `fit` on training data and `transform` test data with the *same* learned numbers. (3) **`fit_transform` is inherited**, written once in the base — add ten more transformers and none of them re-implement it. That single shared method is exactly what `TransformerMixin` gives real sklearn classes.
</details>

## 🧠 Key takeaways

- A class bundles **data + behaviour**; `__init__(self, …)` is the constructor and `self` is the current instance.
- **Instance attributes** are per-object; **class attributes** are shared — keep mutable state per-instance.
- **Methods** use `self` and should call one another to stay DRY and maintainable.
- **Inheritance** reuses a base class and overrides/extends it (`super()`) — for genuine *is-a* relationships only.
- The data-science stack is classes all the way down: **wrappers** like `Dataset`/`DataFrame`, and the **`fit`/`transform`** + **`fit`/`predict`** conventions that power scikit-learn.

## 🚀 Next step

→ **`06_pandas_fundamentals.ipynb`** — DataFrames are objects you instantiate and call methods on. Everything you just learned applies directly.
